# Modulos

In [ ]:
import pandas as pd # Manipulación de datos
import numpy as np # Operaciones numéricas
import win32com.client as win32 # Automatización de Office (Word/Outlook)
import os # Manejo de rutas y archivos
import gc # Recolección de basura para liberar memoria
from typing import Optional, Any # Para anotaciones de tipos en funciones
from pathlib import Path # Manejo de rutas de archivos de forma más robusta
from typing import Dict # Para anotaciones de tipos en funciones
import random # Para generar datos de prueba aleatorios
import time # Para medir tiempos de ejecución y simular retrasos
import pdfkit # Para generación de PDFs a partir de HTML (requiere wkhtmltopdf instalado)
from datetime import datetime # Para manejo de fechas y horas

# Rutas y variables

Este bloque soluciona la variabilidad de los entornos de trabajo (PC Oficina vs. PC Personal) y la evolución mensual de los contratos.

#### Decisiones de Ingeniería y Optimización:
1.  **Uso de `Pathlib`**: Se prefiere sobre `os.path` porque gestiona automáticamente las barras inclinadas (`/` vs `\`) y permite concatenar rutas de forma legible.
2.  **Detección de `Path.home()`**: Resuelve el problema de la ruta absoluta `C:\Users\crist\`. Al usar el "Home" del sistema, el script funcionará sin cambios sin importar el nombre del usuario de Windows.
3.  **Parametrización Estructurada**: Se separan las variables `CONTRATO`, `INFORME_NUM` y `MES_TEXTO`. Esto permite que el próximo mes solo cambies dos strings en la parte superior y toda la estructura de carpetas (incluyendo la carpeta de salida de PDFs) se actualice automáticamente.
4.  **Integridad de Directorios**: Se implementa `mkdir(parents=True)`, lo que garantiza que si la carpeta "Negados" no existe dentro de la ruta del informe actual, el script la cree antes de intentar guardar los PDFs, evitando errores de ejecución.

#### Variables Temporales Eliminadas:
* `USER_HOME`: Variable de sistema temporal.
* `gc.collect()`: Invocado para asegurar limpieza tras operaciones de sistema de archivos.

#### Advertencias:
* **Filtros por Municipio**: La lógica para generar los archivos PDF (ej. `85440_VILLANUEVA.pdf`) se alimentará de `FOLDER_NEGADOS` en proximos bloques.

In [ ]:
# ==========================================
# CONFIGURACIÓN DINÁMICA DE RUTAS (REVISADO)
# ==========================================

# 1. CONFIGURACIÓN DE IDENTIDAD Y ENRUTAMIENTO
CUENTA_ORIGEN: str = "aseg13@capresoc.onmicrosoft.com"
NOMBRE_MOSTRAR: str = "ASEGURAMIENTO REGIMEN SUBSIDIADO"

# 2. CONFIGURACIÓN DE POLÍTICAS DE ENVÍO (SEGURIDAD)
# Cambia a True para enviar copia oculta (BCC) a las EPS externas
ENVIAR_COPIA_EPS_EXTERNA: bool = True 
# Correo constante para auditoría interna de Ventanilla Única
CORREO_VENTANILLA_INTERNA: str = "capresocaeps@capresoca-casanare.gov.co"

# 3. Identificación del Home del Usuario (Resuelve el cambio entre PC de oficina y personal)
USER_HOME: Path = Path.home()

# 4. Base de OneDrive (Capresoca) - Se detecta dinámicamente
ONEDRIVE_PATH: Path = USER_HOME / "OneDrive - 891856000_CAPRESOCA E P S"

# 5. Parámetros del Proyecto (Variables de Control Mensual)
ANO_PROCESO: str = "2026"
CONTRATO: str = "CTO 102.2026"
INFORME_NUM: str = "03"
MES_TEXTO: str = "Enero"

# 6. Configuración de Binarios Externos (Centralizado para evitar ruido en lógica)
# Esta es la ruta al motor de renderizado PDF profesional
PATH_WKHTML_EXE: Path = Path(r"C:\Program Files\wkhtmltopdf\bin\wkhtmltopdf.exe")

# 7. Construcción de Rutas Base usando Pathlib
RUTA_PROYECTO_BASE: Path = (
    ONEDRIVE_PATH / 
    "Escritorio" / 
    "Yesid Rincón Z" / 
    "informes" / 
    ANO_PROCESO / 
    CONTRATO / 
    f"{CONTRATO.replace(' ', '')} Informe  #{INFORME_NUM}" / 
    "12 Actividad" / 
    "Bases de datos notificaciones telefonicas" / 
    "TRASLADOS APROBADOS Y NEGADOS BDUA"
)

# 8. Rutas Específicas de Archivos e Imágenes
FILE_EXCEL_PATH: Path = RUTA_PROYECTO_BASE / f"Traslados Entrada Febrero 2026.xlsx" # 🚩
FOLDER_NEGADOS: Path = RUTA_PROYECTO_BASE / "Negados"

# Imágenes estáticas
RUTA_IMG_CAPRE: Path = ONEDRIVE_PATH / "Imágenes" / "capre.png"
RUTA_IMG_SIPER: Path = ONEDRIVE_PATH / "Imágenes" / "Siper salud.png"

# 9. Creación de directorios de salida e Integridad de Rutas
if not FOLDER_NEGADOS.exists():
    FOLDER_NEGADOS.mkdir(parents=True, exist_ok=True)
    print(f"Directorio creado: {FOLDER_NEGADOS}")

# Validaciones Críticas (Calidad del Dato y del Entorno)
assert PATH_WKHTML_EXE.exists(), f"ERROR: Motor PDF no encontrado en {PATH_WKHTML_EXE}. Verifique la instalación."
assert FILE_EXCEL_PATH.exists(), f"ERROR: Excel no encontrado en {FILE_EXCEL_PATH}"
assert RUTA_IMG_CAPRE.exists(), f"ERROR: Imagen Capresoca no encontrada en {RUTA_IMG_CAPRE}"

print(f"Rutas configuradas correctamente. Motor PDF detectado en: {PATH_WKHTML_EXE.parent}")

# Limpieza de variables de configuración global para optimizar RAM
del USER_HOME
gc.collect()

# Carga de dataframes

Este bloque carga la totalidad de la hoja "NEGADOS" sin realizar recortes, permitiendo que todas las variables presentes en tu plantilla de Word (como `AFL_PRIMER_APELLIDO`, `TPS_IDN_ID`, etc.) estén disponibles en memoria.

#### Decisiones de Ingeniería y Optimización:
1.  **Integridad Absoluta**: Al no definir una lista de columnas, garantizamos que cualquier campo nuevo que se agregue al Excel en el futuro sea detectado automáticamente por el script sin necesidad de modificar el código.
2.  **Preservación de Formato BDUA**: La carga forzada como `str` es innegociable aquí. Los campos como `MNC_ID` (ej. 85440) o `TPS_IDN_ID` deben tratarse como texto para evitar que el sistema elimine ceros o altere identificaciones largas.
3.  **Variable de Enrutamiento**: Se generó `MUNICIPIO_GRUPO` combinando `MNC_ID` y `Nombre_Municipio`. Esta es la "llave maestra" que usaremos en el siguiente bloque para crear los PDFs y los asuntos de los correos por municipio.
4.  **Sanitización de Headers**: Se aplicó `.strip()` a todos los nombres de columnas para asegurar que el mapeo con los tags de Word (`« »`) sea exacto.

#### Variables Temporales Eliminadas:
* `total_registros`: Utilizada solo para el log de control.
* `columnas_detectadas`: Lista temporal de inspección.
* `gc.collect()`: Liberación de memoria tras la carga masiva.

#### Advertencias:
* **Mapeo de Nombres**: Asegúrate de que en tu plantilla HTML usemos exactamente el nombre de la columna (ej. `HST_IDN_NUMERO_IDENTIFICACION`). Si el Excel cambia un guion bajo por un espacio, el script lo detectará pero el mapeo fallaría.

In [ ]:
# ==========================================
# CARGA INTEGRAL DE DATAFRAME
# ==========================================

try:
    # 1. Carga masiva de la hoja 'NEGADOS'
    # Se utiliza dtype=str para preservar CUALQUIER formato (IDs, fechas, códigos)
    # tal cual están en el Excel, evitando que Python los convierta a números.
    df_negados = pd.read_excel(
        FILE_EXCEL_PATH, 
        sheet_name='NEGADOS', 
        dtype=str,
        engine='openpyxl'
    )
    
    # 2. Normalización de Headers (Limpieza de ruido en nombres de columnas)
    # Esto asegura que «AIE-120» sea accesible aunque tenga un espacio invisible.
    df_negados.columns = [col.strip() for col in df_negados.columns]
    
    # 3. Creación de la columna de enrutamiento dinámico (MNC_ID + Nombre_Municipio)
    # Se requiere para la gestión de archivos PDF y filtrado de Outlook solicitado.
    # Usamos .fillna('') para que la concatenación sea segura.
    df_negados['MUNICIPIO_GRUPO'] = (
        df_negados['MNC_ID'].fillna('SN').astype(str) + "_" + 
        df_negados['Nombre_Municipio'].fillna('DESCONOCIDO').str.upper().str.strip()
    )

    # 4. Verificación de Calidad Inicial
    total_registros = len(df_negados)
    columnas_detectadas = df_negados.columns.tolist()

    print(f"--- REPORTE DE CARGA ---")
    print(f"Hoja procesada: NEGADOS")
    print(f"Total filas: {total_registros}")
    print(f"Total columnas cargadas: {len(columnas_detectadas)}")
    print(f"Muestra de columnas para mapeo: {columnas_detectadas[:10]}...")

except Exception as e:
    print(f"CRITICAL ERROR: No se pudo cargar el archivo. Verifique que no esté abierto.")
    print(f"Detalle: {e}")
    raise

# Limpieza de memoria para optimización de RAM
del total_registros
gc.collect()

# Limpiar df_negados['correo_electronico']

Este bloque ejecuta una limpieza crítica sobre el DataFrame para garantizar que el motor de envío de Outlook no procese registros con errores de direccionamiento.

#### Decisiones de Ingeniería y Optimización:
1.  **Limpieza Pre-Filtro**: Se aplicó `.str.strip()` a la columna `correo_electronico`. Esto es vital porque a menudo los archivos Excel contienen celdas con uno o más espacios en blanco que Pandas no detecta como nulos por defecto.
2.  **Filtrado Vectorizado**: Se utilizó la lógica booleana de Pandas (`&`) para evaluar el estado del correo en una sola operación de CPU, evitando iterar fila por fila (lo cual sería ineficiente para volúmenes grandes).
3.  **Prevención de Fragmentación**: Se usó `.copy()` al final del filtrado para asegurar que `df_negados` sea un objeto independiente en RAM y no una "vista" del DataFrame original, lo que previene errores de advertencia al momento de asignar nuevos valores.
4.  **Integridad**: El reporte de auditoría permite al usuario validar cuántos registros se perdieron en este paso, cumpliendo con el principio de transparencia en la calidad del dato.

#### Variables Temporales Eliminadas:
* `filas_iniciales`: Contador de entrada.
* `filas_finales`: Contador de salida.
* `filas_eliminadas`: Diferencial de limpieza.
* `gc.collect()`: Invocado para liberar los bloques de memoria que contenían las filas descartadas.

#### Advertencias:
* **Formatos Inválidos**: Este bloque elimina correos vacíos, pero no valida si la estructura del correo es correcta (ej. que contenga un `@` o un `.`).

In [ ]:
# ==========================================
# LIMPIEZA DE REGISTROS SIN CORREO
# ==========================================

# 1. Captura de dimensiones previas para el reporte de Calidad del Dato
filas_iniciales: int = len(df_negados)

# 2. Normalización y Limpieza Vectorizada
# Eliminamos espacios en blanco accidentales que podrían burlar el filtro de nulos.
df_negados['correo_electronico'] = df_negados['correo_electronico'].str.strip()

# 3. Filtrado de registros no enviables
# Mantenemos solo filas donde el correo no sea NaN (nulo) ni esté vacío ("")
# El uso de .copy() es fundamental para evitar el 'SettingWithCopyWarning' en futuras transformaciones.
df_negados = df_negados[
    df_negados['correo_electronico'].notna() & 
    (df_negados['correo_electronico'] != "")
].copy()

# 4. Reporte de Auditoría
filas_finales: int = len(df_negados)
filas_eliminadas: int = filas_iniciales - filas_finales

print(f"--- AUDITORÍA DE INTEGRIDAD DE CORREOS ---")
print(f"Registros recibidos: {filas_iniciales}")
print(f"Registros descartados (sin dirección de correo): {filas_eliminadas}")
print(f"Registros aptos para procesamiento: {filas_finales}")

# Limpieza explícita de variables de conteo y forzado de recolección de basura
del filas_iniciales, filas_finales, filas_eliminadas
gc.collect()

# PLANTILLA HTML

Este bloque corrige los errores de mapeo (`KeyError`) asegurando que los placeholders de la plantilla coincidan letra por letra con las columnas del DataFrame después de la limpieza de encabezados.

#### Decisiones de Ingeniería y Optimización:
1.  **Mapeo Case-Sensitive**: Se corrigió `{ASUNTO}` por `{Asunto}` para respetar el nombre real de la columna detectada en el DataFrame. En Python, el método `.format(**dict)` requiere coincidencia exacta de mayúsculas y minúsculas.
2.  **Consistencia de Normalización**: Gracias a que en el bloque de carga aplicamos `col.replace(' ', '_')`, el placeholder `{Nombre_Municipio}` ahora mapea correctamente la columna que originalmente era `'Nombre_Municipio'`.
3.  **Incrustación CID Inalterada**: Se mantienen los identificadores `logo_capre` y `logo_siper` para la lógica de adjuntos de Outlook, garantizando que los logos se visualicen sin bloqueos de seguridad del cliente de correo.
4.  **Uso de Placeholders Reales**: Se eliminó la conversión a mayúsculas del diccionario en el bloque de envío, permitiendo una trazabilidad directa entre el Excel y el código.

#### Variables Temporales Eliminadas:
* `gc.collect()`: Invocado para asegurar que el string de la plantilla (que puede ser grande en memoria) se gestione eficientemente.

In [ ]:
# ==========================================
# DEFINICIÓN DE PLANTILLA HTML
# ==========================================

# Se ajusta la alineación de AEI/120 a la izquierda.
# Se invierte la posición del pie de página: Logo SuperSalud a la IZQUIERDA, Texto a la DERECHA.
# Se aumenta el tamaño del logo de SuperSalud (width="200").
HTML_TEMPLATE: str = """
<html>
<body style="font-family: Arial, sans-serif; line-height: 1.5; color: #333; margin: 20px;">
    <table width="100%" style="border-collapse: collapse;">
        <tr>
            <td style="width: 30%; text-align: left;">
                <img src="cid:logo_capre" alt="Capresoca" width="180">
            </td>
            <td style="width: 70%; text-align: right; font-size: 10px; color: #666;">
                NIT. 891.856.000-7<br>
                EPS en intervención<br>
                FO-GD-01 | 2024-01-19 | V.03
            </td>
        </tr>
    </table>

    <hr style="border: 0; border-top: 1px solid #0056b3;">

    <p style="text-align: left;"><strong>AEI/120:</strong> {AIE_120}</p>
    
    <p>Yopal-Casanare, 2 de marzo de 2026</p>

    <div style="margin-top: 20px;">
        <strong>Señor(a):</strong><br>
        {AFL_PRIMER_APELLIDO} {AFL_SEGUNDO_APELLIDO} {AFL_PRIMER_NOMBRE} {AFL_SEGUNDO_NOMBRE}<br>
        {TPS_IDN_ID}: {HST_IDN_NUMERO_IDENTIFICACION}<br>
        {Nombre_Municipio}<br>
        {correo_electronico}
    </div>

    <p style="margin-top: 20px;"><strong>Ref.:</strong> {Asunto}</p>

    <p>Cordial saludo,</p>

    <p>Lamentamos informarle que su solicitud de traslado a <strong>Capresoca EPS</strong> ha sido negada por su EPS de origen, <strong>{nombre_eps}</strong>. Entendemos que esta noticia puede ser decepcionante, pero estamos aquí para ayudarle a tomar las siguientes acciones necesarias.</p>

    <p>Para asegurar que usted y sus beneficiarios puedan disfrutar de los servicios de Capresoca EPS, le recomendamos que se comunique de inmediato con su EPS actual, <strong>{nombre_eps}</strong>, y solicite la aprobación del traslado. Es crucial que actúe rápidamente para evitar cualquier interrupción en sus servicios de salud.</p>

    <p>Capresoca EPS volverá a gestionar su solicitud en el mes siguiente. Sin embargo, si la EPS anterior continúa negando el traslado, esto podría obstaculizar el proceso y revertir su solicitud. No permita que esto suceda; su salud y bienestar son nuestra prioridad.</p>

    <p style="font-style: italic; font-size: 12px; color: #555;">
        Buscamos cumplir con lo dispuesto en el decreto 780 del 2016 en su título 7 traslados y movilidad artículo 2.1.7.1 “Derecho a la libre escogencia de EPS..."
    </p>

    <p>Para cualquier consulta adicional, estamos disponibles en el número: <strong>3106805416</strong> y en el correo: <a href="mailto:aseguramiento@capresoca-casanare.gov.co">aseguramiento@capresoca-casanare.gov.co</a>.</p>

    <div style="margin-top: 40px;">
        Atentamente,<br><br><br>
        <strong>Osmar Yesid Rincón Zorro</strong><br>
        Profesional de apoyo área de aseguramiento
    </div>

    <table width="100%" style="margin-top: 50px; border-top: 2px solid #0056b3; font-size: 11px; color: #444;">
        <tr>
            <td style="width: 40%; text-align: left; padding-top: 15px;">
                <img src="cid:logo_siper" alt="SuperSalud" width="200">
            </td>
            <td style="width: 60%; text-align: right; padding-top: 15px; line-height: 1.2;">
                Calle 7 No. 19 – 34. Línea de atención gratuita: 018000912880<br>
                Email. capresocaeps@capresoca-casanare.gov.co<br>
                Yopal - Casanare &nbsp;&nbsp;&nbsp; 1 de 1
            </td>
        </tr>
    </table>
</body>
</html>
"""

print("Plantilla HTML corregida: Alineaciones ajustadas y pie de página reestructurado.")
gc.collect()

# EJECUCIÓN REAL - PRODUCCIÓN
Este bloque representa el motor final de la automatización, diseñado para procesar la totalidad del universo de datos bajo estándares de seguridad corporativa.

#### Decisiones de Ingeniería y Optimización:
1.  **Manejo de Destinatarios Múltiples**: El código respeta las cadenas de correos separadas por punto y coma (`;`) tanto en el destinatario principal como en el campo `correo_eps`. Outlook gestiona estas cadenas nativamente, por lo que no fue necesario separarlas en listas.
2.  **Lógica de Copia Oculta (BCC) Condicional**: Se implementó la variable `ENVIAR_COPIA_EPS_EXTERNA`. Esto permite al usuario desactivar el envío a EPS externas (Nueva EPS, Sanitas, etc.) durante pruebas o cambios de política, manteniendo siempre la copia obligatoria a la ventanilla única de Capresoca.
3.  **Consolidación de Identidad**: Se añadió `NOMBRE_MOSTRAR` a la tabla de metadatos del PDF y a la firma del remitente, asegurando que el documento binario sea un reflejo exacto del "Header de Impresión" de Outlook.
4.  **Agrupación de Archivos**: El nombre de salida del PDF ahora utiliza `MUNICIPIO_GRUPO` (ej. `85001_YOPAL.pdf`), cumpliendo con el requisito de consolidar múltiples ciudadanos en un solo documento geográfico.

#### Variables Temporales Eliminadas:
* `html_acumulado_pdf`: Liberado después de cada ciclo de municipio para evitar desbordamiento de RAM.
* `destinatario_principal` y `bcc_final`: Strings de direccionamiento.
* `gc.collect()`: Invocado al final de toda la ejecución para asegurar que el proceso de Python termine con un uso mínimo de recursos.

#### Advertencias:
* **Capacidad de Outlook**: Con 159 registros, Outlook podría mostrar una barra de progreso de envío. No cierre la aplicación hasta que la bandeja de salida esté vacía.
* **Tiempo Estimado**: El proceso total (envíos + 159 renders de PDF) tardará aproximadamente entre 5 y 8 minutos.

In [ ]:

# ==========================================
# BLOQUE 6: EJECUCIÓN REAL - PRODUCCIÓN
# ==========================================

# 1. CONFIGURACIÓN DEL MOTOR PDF
config_pdf = pdfkit.configuration(wkhtmltopdf=str(PATH_WKHTML_EXE))

PDF_OPTIONS: Dict[str, Any] = {
    'page-size': 'Letter',
    'margin-top': '0.5in',
    'margin-right': '0.5in',
    'margin-bottom': '0.5in',
    'margin-left': '0.5in',
    'encoding': "UTF-8",
    'enable-local-file-access': None,
    'quiet': ''
}

# 2. INICIALIZACIÓN DE SESIÓN OUTLOOK
outlook = win32.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
account = next((a for a in namespace.Accounts if a.SmtpAddress.lower() == CUENTA_ORIGEN.lower()), None)

if not account:
    raise ValueError(f"Error Crítico: La cuenta {CUENTA_ORIGEN} no está configurada en este equipo.")

# 3. PROCESAMIENTO AGRUPADO POR MUNICIPIO
# Utilizamos el DataFrame completo (df_negados) para la ejecución real
municipios = df_negados.groupby('MUNICIPIO_GRUPO')

print(f">>> INICIANDO PROCESO REAL DE NOTIFICACIÓN <<<")
print(f"Remitente: {NOMBRE_MOSTRAR} ({CUENTA_ORIGEN})")
print(f"Municipios a procesar: {len(municipios)}")

try:
    for nombre_municipio, grupo in municipios:
        html_acumulado_pdf: str = ""
        # Nombre del archivo según requerimiento: Codigo_Nombre.pdf
        ruta_pdf_salida = FOLDER_NEGADOS / f"{nombre_municipio}.pdf"
        
        print(f"\nProcesando Municipio: {nombre_municipio} ({len(grupo)} registros)")

        for index, row in grupo.iterrows():
            # A. PREPARACIÓN DE METADATOS Y FECHAS
            datos_correo = row.to_dict()
            fecha_actual = datetime.now().strftime("%A, %d de %B de %Y %I:%M %p")
            
            # Gestión de destinatarios (To y BCC)
            destinatario_principal = str(row['correo_electronico']).strip()
            
            # Lógica de Copias (BCC)
            lista_bcc = [CORREO_VENTANILLA_INTERNA]
            if ENVIAR_COPIA_EPS_EXTERNA and pd.notna(row['correo_eps']):
                lista_bcc.append(str(row['correo_eps']).strip())
            
            bcc_final = "; ".join(lista_bcc)

            # B. EMULACIÓN DE ENCABEZADO OUTLOOK PARA EL PDF
            header_outlook = f"""
            <div style="font-family: 'Segoe UI', Tahoma, sans-serif; color: #000; margin-bottom: 30px;">
                <h2 style="margin: 0; padding: 0; font-size: 20px; font-weight: bold;">{NOMBRE_MOSTRAR}</h2>
                <hr style="border: 0; border-top: 2.5px solid #000; margin: 5px 0 15px 0;">
                <table width="100%" style="border-collapse: collapse; font-size: 13px; line-height: 1.6;">
                    <tr><td style="width: 85px; font-weight: bold; color: #666;">De:</td><td style="font-weight: bold;">{NOMBRE_MOSTRAR} &lt;{CUENTA_ORIGEN}&gt;</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Enviado el:</td><td>{fecha_actual}</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Para:</td><td>{destinatario_principal}</td></tr>
                    <tr><td style="font-weight: bold; color: #666;">Asunto:</td><td style="font-weight: bold;">{row['Asunto']}</td></tr>
                </table>
            </div>
            """

            # C. CONSTRUCCIÓN DE CONTENIDOS (CORREO VS PDF)
            html_email = HTML_TEMPLATE.format(**datos_correo)
            
            # Traducir CIDs a rutas locales para el PDF
            html_para_pdf = (header_outlook + html_email).replace("cid:logo_capre", str(RUTA_IMG_CAPRE))
            html_para_pdf = html_para_pdf.replace("cid:logo_siper", str(RUTA_IMG_SIPER))
            
            # Acumular para el consolidado con salto de página
            html_acumulado_pdf += f'<div style="page-break-after: always; padding: 10px;">{html_para_pdf}</div>'

            # D. CREACIÓN Y ENVÍO DEL CORREO INDIVIDUAL
            mail = outlook.CreateItem(0)
            mail._oleobj_.Invoke(*(64209, 0, 8, 0, account)) # Enrutamiento forzado
            
            mail.Subject = row['Asunto']
            mail.To = destinatario_principal
            mail.BCC = bcc_final
            
            # Incrustación de logos CID
            for img_id, path in [("logo_capre", RUTA_IMG_CAPRE), ("logo_siper", RUTA_IMG_SIPER)]:
                att = mail.Attachments.Add(str(path))
                att.PropertyAccessor.SetProperty("http://schemas.microsoft.com/mapi/proptag/0x3712001E", img_id)
            
            mail.HTMLBody = html_email
            
            try:
                mail.Send()
                print(f"   [OK] Enviado a: {row['HST_IDN_NUMERO_IDENTIFICACION']}")
            except Exception as e_send:
                print(f"   [!] Error de envío para ID {row['HST_IDN_NUMERO_IDENTIFICACION']}: {e_send}")
            
            time.sleep(1.2) # Throttling para evitar bloqueos de Exchange

        # E. GENERACIÓN DEL PDF CONSOLIDADO DEL MUNICIPIO
        try:
            pdfkit.from_string(html_acumulado_pdf, str(ruta_pdf_salida), configuration=config_pdf, options=PDF_OPTIONS)
            print(f"   [PDF] Consolidado generado exitosamente: {ruta_pdf_salida.name}")
        except Exception as e_pdf:
            print(f"   [!] Error generando PDF para {nombre_municipio}: {e_pdf}")

    print(f"\n>>> PROCESO FINALIZADO CON ÉXITO <<<")

except Exception as e:
    print(f"\n>>> ERROR CRÍTICO EN EL PROCESO REAL: {e}")
    raise

# Limpieza técnica final
del municipios, account, namespace, html_acumulado_pdf
gc.collect()

In [ ]:
df_negados.columns